In [ ]:
import ee

try:
    ee.Initialize(project="viirs-peru")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="viirs-peru")

print("Google Earth Engine listo")

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

def find_project_root(marker="requirements.txt"):
    path = Path.cwd()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"No se encontró '{marker}' subiendo desde {path}")

PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data"
RAW_RENAMU = DATA / "raw" / "RENAMU"

geobase = gpd.read_file(DATA / "clean" / "staging" / "geobase_distrital.gpkg")
ubigeos_validos = set(geobase["UBIGEO"])

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Carpeta RENAMU: {RAW_RENAMU}")
print(f"Distritos válidos en geobase: {len(ubigeos_validos)}")

# Datos administrativos y municipales

# Bitácora — causal-anemia-model, notebook 00g_renamu (data/01_database)

Documentación de la sesión de trabajo sobre `00g_renamu.ipynb`, continuación de la bitácora de los notebooks 00a-00f.

## Fuente de datos

**RENAMU (Registro Nacional de Municipalidades)** — INEI. https://proyectos.inei.gob.pe/microdatos/ (buscar "RENAMU"). Encuesta anual que aplica el INEI a todas las municipalidades del país. Desde 2021 se estandariza en un único módulo "Registro Nacional de Municipalidades - RENAMU", sin la división por módulos separados que tenía en años anteriores.

En este proyecto, RENAMU se usa como fuente de variables de control para el modelo causal (Etapa 6, DML + Causal Forest). El riesgo que cubre: que el modelo confunda "el gasto no rinde porque el territorio no responde" con "el gasto no rinde porque la municipalidad gestiona mal". Sin este control, τ podría estar capturando capacidad de gestión municipal en vez de contexto territorial — que es justo lo que el proyecto busca aislar.

## Por qué el panel arranca en 2021 y no en 2020

- El módulo con la pregunta sobre anemia recién aparece en la encuesta 2021. RENAMU 2020 no la tiene.
- Desde 2021 el formato está estandarizado en un único módulo: mismos nombres de columna año a año.
- Desde 2021 el propio CSV trae la columna `Ubigeo` ya limpia en la cabecera. En 2020 solo existe `idmunici`, y habría que reconstruir el ubigeo a mano — trabajo extra que no vale la pena si igual ese año no trae la variable que más importa.

Por eso el panel se arma como loop 2021 → último año disponible, con el mismo código para cada año.

## Variables extraídas (panel por ubigeo + Año)

| Variable | Código | Nombre final | Descripción (diccionario RENAMU 2021) | Corrección de año |
|---|---|---|---|---|
| Personal municipal | P19D_T | `personal_total` | "Total personal / 31 de diciembre 2020" — Módulo III, pregunta 19. Conteo, no Sí/No. | Retrospectiva: año real = Año_encuesta − 1 |
| Programa de prevención de anemia con MINSA | **P68_10** | `programa_anemia` | Módulo V - Salud, pregunta 59: "En el año 2020, ¿La municipalidad implementó programas de control y prevención de la salud en coordinación con el MINSA en: Prevención y reducción de la anemia". Ítem dentro de checklist P68_1 a P68_11. Código de "Sí" = 10. | Retrospectiva: año real = Año_encuesta − 1 |
| Centro de salud administrado por la municipalidad | P66_2 | `centro_salud_municipal` | Módulo V - Salud, pregunta 57: "¿En el Distrito funcionan establecimientos de salud administrados por la municipalidad: Centro de salud?" — 1: Sí / 2: No. Pregunta en tiempo presente. | No lleva corrección: año real = Año_encuesta |

## Hallazgo importante: corrección del código de `programa_anemia` (P68_7 → P68_10)

En una iteración anterior del notebook, `programa_anemia` estaba mapeado a la columna `P68_7`, asumiendo que el código de "Sí" era 7. Al revisar el diccionario de datos oficial (`Diccionario_Anexo01.pdf`, RENAMU 2021, Módulo V - Salud, pregunta 59) se confirmó que **P68_7 no es la variable de anemia**: es un checklist de 9 ítems (más 2 adicionales fuera de secuencia), cada uno con su propio código de "Sí":

- P68_1 → Vacunación → 1: Sí
- P68_2 → Control de crecimiento y desarrollo del niño → 2: Sí
- P68_3 → Control de infecciones respiratorias agudas → 3: Sí
- P68_4 → Control de enfermedades diarreicas agudas → 4: Sí
- P68_5 → Control de tuberculosis → 5: Sí
- P68_6 → Planificación familiar → 6: Sí
- **P68_7 → Control de infecciones de transmisión sexual y VIH/SIDA → 7: Sí** (esto es lo que estaba mal mapeado como anemia)
- P68_8 → Otro → 8: Sí
- P68_9 → No implementó programas → 9: Sí
- **P68_10 → Prevención y reducción de la anemia → 10: Sí** (código correcto)
- P68_11 → Preparación y respuesta frente al COVID-19 → 11: Sí

Se corrigió el mapeo a `P68_10`, y la recodificación booleana pasó de `== 7` a `== 10`. Este hallazgo se hizo revisando el PDF del diccionario oficial antes de cerrar la recodificación — evitó que el modelo terminara usando la variable equivocada (VIH/SIDA en vez de anemia) sin ningún error visible en el código.

**Pendiente de confirmar:** el usuario indica que los diccionarios de 2022, 2023, 2024 y 2025 mantienen las mismas variables que 2021. Falta verificar empíricamente con `value_counts()` sobre cada año cargado que el código 10 = Sí se mantiene igual en todos los años del panel, antes de dar por cerrada la recodificación completa.

## Decisión de diseño: una sola base, no tres tablas separadas

Se consideró inicialmente dividir el panel en tres tablas separadas (una por variable), cada una con su propio `Año` ya corregido (Año_encuesta − 1 para las dos variables retrospectivas), listas para mergear directamente contra SIEN y SIAF. Se descartó esa opción a favor de una sola tabla, por dos razones:

1. **Consistencia con la convención ya establecida** en los notebooks 00b-00f: un archivo de salida por fuente de datos, sin fragmentar.
2. **Simplicidad:** las tres tablas no aportaban información nueva, solo reorganizaban el mismo panel — se pueden derivar en el momento en que se arme la tabla_maestra, sin necesidad de mantener archivos redundantes en `data/clean/staging/`.

La tabla final usa **un solo `Año`** (= `Año_encuesta`, el año en que se aplicó la encuesta) como llave, documentado explícitamente en el código: `personal_total` y `programa_anemia` describen en realidad el año anterior a ese `Año`, y `centro_salud_municipal` describe el año de la encuesta directamente. Esta corrección de desfase se debe aplicar en el notebook donde se construya la `tabla_maestra` y se haga el merge con SIEN (anemia) y SIAF (gasto) — no en `00g_renamu.ipynb`.

## Convenciones aplicadas (heredadas de 00a-00f)

- Bloque `find_project_root()` / `PROJECT_ROOT` / `DATA` en vez de rutas hardcodeadas (`"../data/raw/..."`), que se rompían según la subcarpeta del notebook.
- `geobase_distrital.gpkg` como fuente única de verdad geográfica: se cargan los 1889 UBIGEO válidos y se filtran los registros de RENAMU que no aparecen en la geobase (excluidos por el mismo criterio documentado en 00a).
- `ee.Initialize(project="viirs-peru")` se dejó en la primera celda por consistencia con los demás notebooks de la carpeta, aunque RENAMU no usa Earth Engine (es solo lectura de CSVs administrativos) — es opcional, no afecta el resto del notebook si se elimina.
- Notebook organizado en celdas separadas: setup/imports → configuración y función de carga → loop de descarga (solo eso) → consolidar → validar contra geobase → recodificar → guardar.
- Output guardado en `data/clean/staging/`, versionado en git (a diferencia de `data/raw/`, que sigue ignorado).

## Estructura del notebook `00g_renamu.ipynb`

1. Earth Engine init (opcional, boilerplate heredado de los demás notebooks).
2. Rutas robustas + imports + carga de `geobase_distrital.gpkg`.
3. Diccionario `columnas_necesarias` (con `P68_10` corregido) + función `cargar_renamu_año()` reutilizable, con diagnóstico de valores únicos de `programa_anemia` por año.
4. Loop de descarga (una celda dedicada, para no perder el trabajo de consolidación si algún año falla).
5. Consolidación en un panel único, renombrando `Año_encuesta` → `Año`.
6. Validación de ubigeos contra la geobase (exclusión de los no reconocidos).
7. Revisión de valores únicos de `programa_anemia` (diagnóstico) + recodificación booleana (`programa_anemia == 10`, `centro_salud_municipal == 1`, `personal_total` a `Int64`).
8. Guardado final.

## Output

- **Archivo:** `data/clean/staging/renamu_distrital_2021_2024.csv` (nombre del rango de años ajustable según disponibilidad real en el portal).
- **Columnas:** `ubigeo`, `Año`, `personal_total`, `programa_anemia`, `centro_salud_municipal`.
- **Grano:** una fila por municipalidad (ubigeo) por año de encuesta.

## Pendiente para la siguiente sesión

- Correr el notebook completo y confirmar, con `value_counts()` por año, que el código 10 = Sí de `programa_anemia` se mantiene igual en 2022, 2023, 2024 y 2025.
- Confirmar el rango real de años disponibles en el portal de microdatos del INEI (el `range(2021, 2025)` es un supuesto de partida).
- Al construir la `tabla_maestra` (notebook futuro de merge): aplicar la corrección de año (Año − 1) a `personal_total` y `programa_anemia` antes de cruzar con SIEN y SIAF; usar `Año` sin corrección para `centro_salud_municipal`.
- `00h_sien_reunis.ipynb` y `00i_siaf_gasto_devengado.ipynb` siguen sin revisar — les corresponde el mismo tratamiento de convenciones (rutas robustas, geobase como fuente única de verdad, guardado en `data/clean/staging/`).

## Otros temas resueltos en la sesión

- Se aclaró cómo agregar colaboradores al proyecto de Google Cloud "viirs-peru" (IAM → Otorgar acceso → rol "Earth Engine Resource Writer"), y que además cada colaborador necesita su propia cuenta de Google registrada en Earth Engine (code.earthengine.google.com/register), independientemente del rol IAM.
- Se diagnosticó un error "Invalid request" en la pantalla de generación de token de `ee.Authenticate()` para un colaborador ya registrado: causas más probables son una versión desactualizada de `earthengine-api` (`pip install earthengine-api --upgrade`) o necesidad de forzar el modo de autenticación (`ee.Authenticate(auth_mode='localhost')` o vía terminal con `earthengine authenticate`), no un problema de permisos IAM.

In [ ]:
años = range(2021, 2025)  # ajustar según lo que confirmes disponible en el portal

columnas_necesarias = {
    "Ubigeo": "ubigeo",
    "P19D_T": "personal_total",
    "P68_10": "programa_anemia",       # corregido: antes decía P68_7 (esa era VIH/SIDA, no anemia)
    "P66_2": "centro_salud_municipal",
}

nulos_renamu = ["#Â¡NULO!", "#¡NULO!"]


def cargar_renamu_año(año, base_dir=RAW_RENAMU):
    """Carga y valida el CSV de RENAMU de un año. Devuelve None si algo falla."""
    carpeta = base_dir / str(año)

    if not carpeta.exists():
        print(f"⚠️ {año}: no existe la carpeta {carpeta}, saltando")
        return None

    csvs = [f for f in carpeta.iterdir() if f.suffix.lower() == ".csv"]
    if len(csvs) != 1:
        print(f"⚠️ {año}: encontré {len(csvs)} CSV en la carpeta, revisar manualmente")
        return None

    df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)

    faltantes = [c for c in columnas_necesarias if c not in df.columns]
    if faltantes:
        print(f"⚠️ {año}: faltan columnas {faltantes} — revisar nombre exacto en el diccionario {año}")
        print(f"   Columnas disponibles (primeras 15): {list(df.columns)[:15]}")
        return None

    df = df[list(columnas_necesarias.keys())].rename(columns=columnas_necesarias)
    df["ubigeo"] = df["ubigeo"].astype(str).str.zfill(6)
    df["Año_encuesta"] = año

    print(f"{año} — valores únicos en programa_anemia: {df['programa_anemia'].unique()}")
    print(f"✅ {año}: {df.shape[0]} municipalidades cargadas")

    return df

In [ ]:
paneles = [cargar_renamu_año(año) for año in años]
paneles = [df for df in paneles if df is not None]

In [ ]:
renamu_panel = pd.concat(paneles, ignore_index=True)

# IMPORTANTE — corrección de año pendiente para el merge futuro:
# personal_total (P19D_T) y programa_anemia (P68_10) son retrospectivas: describen
# el año ANTERIOR a Año_encuesta, no el año de la encuesta en sí.
# centro_salud_municipal (P66_2) sí describe el año de la encuesta directamente.
# Esta tabla se guarda con un solo "Año" (= Año_encuesta) como llave — el ajuste
# de -1 para personal_total y programa_anemia se aplica en el notebook donde se
# arme la tabla_maestra y se haga el merge con SIEN/SIAF, no aquí.

renamu_panel = renamu_panel.rename(columns={"Año_encuesta": "Año"})

print(renamu_panel.shape)
renamu_panel.head()

In [ ]:
no_reconocidos = sorted(set(renamu_panel["ubigeo"]) - ubigeos_validos)
if no_reconocidos:
    print(f"⚠️ {len(no_reconocidos)} ubigeos en RENAMU no están en la geobase (se excluyen): {no_reconocidos[:10]}")

renamu_panel = renamu_panel[renamu_panel["ubigeo"].isin(ubigeos_validos)].copy()
print(renamu_panel.shape)

In [ ]:
print(renamu_panel["programa_anemia"].value_counts(dropna=False))

## Recodificacion

In [ ]:
renamu_panel["programa_anemia"] = (renamu_panel["programa_anemia"] == 10).astype("Int64")
renamu_panel["centro_salud_municipal"] = (renamu_panel["centro_salud_municipal"] == 1).astype("Int64")
renamu_panel["personal_total"] = renamu_panel["personal_total"].astype("Int64")

print(renamu_panel.dtypes)
renamu_panel.head()

## guardar

In [ ]:
output_dir = DATA / "clean" / "staging"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "renamu_distrital_2021_2024.csv"
renamu_panel.to_csv(output_path, index=False)

print(f"Guardado: {output_path}")

### Notas de codificación y tipos de dato

**`centro_salud_municipal` (P66_2):** es un indicador binario (Sí/No), no una cantidad. Corresponde específicamente al tipo de establecimiento "Centro de salud" administrado por la municipalidad — no incluye hospitales, postas, consultorios ni otros tipos (esos son preguntas separadas en el diccionario: P66_1, P66_3, P66_4, etc.). El campo de conteo real (`P66_2_1`, "Número de establecimientos") no forma parte de este panel.

**`programa_anemia` (P68_7):** el diccionario de RENAMU codifica esta pregunta como parte de un checklist de 11 programas de salud (P68_1 a P68_11), donde cada ítem usa su propia posición como código de "Sí" en vez de un 1/2 estándar. Para P68_7 específicamente: `0 = Pase` (no marcó esta opción) y `7 = Sí` (sí implementó el programa de anemia). Antes de usar esta variable en el modelo causal, se recodifica a booleano estándar (1 = Sí, 0 = No) para que no se interprete como una magnitud numérica.

**Tipos de dato:** las tres variables llegan como `float64` por los valores nulos (`NaN`) presentes en el CSV crudo. Se mantienen como `float64` hasta el merge final — convertir a `int` antes de imputar/tratar los nulos generaría un error, porque `NaN` no es representable como entero en pandas.